# Chapter 11 - HMM Exercise 2: Predicting Student Health States

## Scenario
A university health center is monitoring the well-being of a student over **4 days**. The student's true health condition is hidden, but each day the center can observe certain behavioral signals.

You will model this situation using a **Hidden Markov Model (HMM)**.

### Hidden States (Health Conditions)
- `healthy`
- `tired`
- `sick`

### Observations (Daily Behaviors)
- `attend_class`
- `sleep_early`
- `skip_class`
- `visit_clinic`

### Initial Probabilities
- P(healthy) = 0.5
- P(tired) = 0.3
- P(sick) = 0.2

### Transition Probabilities
| From → To | healthy | tired | sick |
|-----------|---------|-------|------|
| healthy   | 0.6     | 0.3   | 0.1  |
| tired     | 0.2     | 0.5   | 0.3  |
| sick      | 0.1     | 0.3   | 0.6  |

### Emission Probabilities
| State → Observation | attend_class | sleep_early | skip_class | visit_clinic |
|--------------------|--------------|-------------|------------|--------------|
| healthy            | 0.5          | 0.3         | 0.1        | 0.1          |
| tired              | 0.3          | 0.4         | 0.2        | 0.1          |
| sick               | 0.1          | 0.2         | 0.3        | 0.4          |

### Observed Sequence (4 Days)
`O = ["attend_class", "sleep_early", "skip_class", "visit_clinic"]`

---
## Tasks
1. Implement the Viterbi algorithm.
2. Compute the most likely health-state sequence over the 4 days.
3. Print the final hidden-state path.
4. Predict state distributions for Day 5 and Day 6.

In [1]:
# Define model components
states = ["healthy", "tired", "sick"]
observations = ["attend_class", "sleep_early", "skip_class", "visit_clinic"]

pi = {"healthy": 0.5, "tired": 0.3, "sick": 0.2}

A = {
    "healthy": {"healthy": 0.6, "tired": 0.3, "sick": 0.1},
    "tired":   {"healthy": 0.2, "tired": 0.5, "sick": 0.3},
    "sick":    {"healthy": 0.1, "tired": 0.3, "sick": 0.6}
}

B = {
    "healthy": {"attend_class": 0.5, "sleep_early": 0.3, "skip_class": 0.1, "visit_clinic": 0.1},
    "tired":   {"attend_class": 0.3, "sleep_early": 0.4, "skip_class": 0.2, "visit_clinic": 0.1},
    "sick":    {"attend_class": 0.1, "sleep_early": 0.2, "skip_class": 0.3, "visit_clinic": 0.4}
}

O = ["attend_class", "sleep_early", "skip_class", "visit_clinic"]

states, O

(['healthy', 'tired', 'sick'],
 ['attend_class', 'sleep_early', 'skip_class', 'visit_clinic'])

In [2]:
#load libraries
import pandas as pd

# 1. Viterbi algorithm implementation
def viterbi(O, states, pi, A, B):
    # V[t][s] lưu xác suất cao nhất để đi đến trạng thái s tại thời điểm t
    V = [{}]

    # path[s] lưu chuỗi trạng thái tốt nhất dẫn đến trạng thái s
    path = {}

    # Bước khởi tạo (t = 0)
    for s in states:
        # V_0(s) = pi[s] * B[s][O[0]]
        V[0][s] = pi[s] * B[s][O[0]]
        path[s] = [s]

    # Bước đệ quy: tính từ t = 1 đến t = len(O)-1
    for t in range(1, len(O)):
        V.append({})
        new_path = {}

        for j in states:
            # Tìm trạng thái i* ở t-1 cho xác suất: V[t-1][i] * A[i][j] lớn nhất
            (prob, best_state) = max((V[t-1][i] * A[i][j], i) for i in states)

            # Nhân thêm xác suất phát B[j][O[t]]
            V[t][j] = prob * B[j][O[t]]

            # Cập nhật đường đi tốt nhất dẫn tới j
            new_path[j] = path[best_state] + [j]

        path = new_path

    # Bước kết thúc: chọn trạng thái kết thúc có xác suất cao nhất
    final_state = max(states, key=lambda s: V[-1][s])

    return V, final_state, path

# 2. Compute the most likely weather sequence for the 4 days.
V, final_state, path_dict = viterbi(O, states, pi, A, B)

df = pd.DataFrame(V)
df.index = ["Day 1", "Day 2", "Day 3", "Day 4"]
df

,healthy,tired,sick
Day 1,0.250000,0.09000,0.020000
Day 2,0.045000,0.03000,0.005400
Day 3,0.002700,0.00300,0.002700
Day 4,0.000162,0.00015,0.000648


In [3]:
# 3. Print final most likely hidden-state sequence
print("Most likely hidden state sequence:", path_dict[final_state])

Most likely hidden state sequence: ['healthy', 'tired', 'sick', 'sick']


In [7]:
# 4. Predict state probabilities for Day 5 and Day 6
import numpy as np

# Chuyển ma trận A sang dạng numpy
A_mat = np.array([
    [A["healthy"]["healthy"], A["healthy"]["tired"], A["healthy"]["sick"]],
    [A["tired"]["healthy"], A["tired"]["tired"], A["tired"]["sick"]],
    [A["sick"]["healthy"], A["sick"]["tired"], A["sick"]["sick"]]
])

# Lấy phân phối trạng thái ở ngày 4 từ bảng Viterbi
# Ta chuẩn hóa xác suất ở ngày cuối để thành phân phối hợp lệ
last_day = V[-1]
posterior_day4 = np.array([last_day["healthy"], last_day["tired"], last_day["sick"]], dtype=float)
posterior_day4 = posterior_day4 / posterior_day4.sum()

# Dự đoán ngày 5
posterior_day5 = posterior_day4 @ A_mat
print("Day 5 state probabilities (healthy, tired, sick):", posterior_day5)

# Dự đoán ngày 6
posterior_day6 = posterior_day5 @ A_mat
print("Day 6 state probabilities (healthy, tired, sick):", posterior_day6)

Day 5 state probabilities (healthy, tired, sick): [0.2     0.33125 0.46875]
Day 6 state probabilities (healthy, tired, sick): [0.233125 0.36625  0.400625]
